In [1]:
import numpy as np
import pandas as pd 
import matplotlib as plt
import math
import random
import csv
import json

In [2]:
path_deliveries = "C:\\Users\\pmoh3005\\OneDrive - 7-Eleven, Inc\\Desktop\\development\\7-11apps\\IPL\\archive\\deliveries.csv"
path_matches = "C:\\Users\\pmoh3005\\OneDrive - 7-Eleven, Inc\\Desktop\\development\\7-11apps\\IPL\\archive\\matches.csv"

# Task 0 : load data

In [27]:
# numpy load gen txt
deliveries = np.genfromtxt(path_deliveries,delimiter=",",dtype=None,names=True,encoding="utf-8")
print(deliveries[:5])

[(335982, 1, 'Kolkata Knight Riders', 'Royal Challengers Bangalore', 0, 1, 'SC Ganguly', 'P Kumar', 'BB McCullum', 0, 1, 1, 'legbyes', 0, 'NA', 'NA', 'NA')
 (335982, 1, 'Kolkata Knight Riders', 'Royal Challengers Bangalore', 0, 2, 'BB McCullum', 'P Kumar', 'SC Ganguly', 0, 0, 0, '', 0, 'NA', 'NA', 'NA')
 (335982, 1, 'Kolkata Knight Riders', 'Royal Challengers Bangalore', 0, 3, 'BB McCullum', 'P Kumar', 'SC Ganguly', 0, 1, 1, 'wides', 0, 'NA', 'NA', 'NA')
 (335982, 1, 'Kolkata Knight Riders', 'Royal Challengers Bangalore', 0, 4, 'BB McCullum', 'P Kumar', 'SC Ganguly', 0, 0, 0, '', 0, 'NA', 'NA', 'NA')
 (335982, 1, 'Kolkata Knight Riders', 'Royal Challengers Bangalore', 0, 5, 'BB McCullum', 'P Kumar', 'SC Ganguly', 0, 0, 0, '', 0, 'NA', 'NA', 'NA')]


In [41]:
# numpy load gen txt
matches = np.genfromtxt(
    path_matches,
    delimiter=',',
    usecols = range(0,20)
)
print(matches[:5])

[[        nan         nan         nan         nan         nan         nan
          nan         nan         nan         nan         nan         nan
          nan         nan         nan         nan         nan         nan
          nan         nan]
 [3.35982e+05         nan         nan         nan         nan         nan
          nan         nan         nan         nan         nan         nan
          nan 1.40000e+02 2.23000e+02 2.00000e+01         nan         nan
          nan         nan]
 [3.35983e+05         nan         nan         nan         nan         nan
          nan         nan         nan         nan         nan         nan
          nan         nan 3.30000e+01 2.41000e+02 2.00000e+01         nan
          nan         nan]
 [3.35984e+05         nan         nan         nan         nan         nan
          nan         nan         nan         nan         nan         nan
          nan 9.00000e+00 1.30000e+02 2.00000e+01         nan         nan
          nan         nan]
 [3.

# Columns for ref

In [5]:
print(deliveries.dtype.names)
match_id = deliveries["match_id"]
inning = deliveries["inning"]
batting_team = deliveries["batting_team"]
bowling_team = deliveries["bowling_team"]
over = deliveries["over"]
ball = deliveries["ball"]
batter = deliveries["batter"]
bowler = deliveries["bowler"]
non_striker = deliveries["non_striker"]
batsman_runs = deliveries["batsman_runs"]
extra_runs = deliveries["extra_runs"]
total_runs = deliveries["total_runs"]
extras_type = deliveries["extras_type"]
is_wicket = deliveries["is_wicket"]
player_dismissed = deliveries["player_dismissed"]
dismissal_kind = deliveries["dismissal_kind"]
fielder = deliveries["fielder"]

('match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball', 'batter', 'bowler', 'non_striker', 'batsman_runs', 'extra_runs', 'total_runs', 'extras_type', 'is_wicket', 'player_dismissed', 'dismissal_kind', 'fielder')


# Generalizing team names pre processing

In [6]:
name_map = {
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru',
    'Rising Pune Supergiant': 'Rising Pune Supergiants'
}
batting_team = np.array([name_map.get(t,t) for t in batting_team])

# Task 1 : Total runs per match

In [ ]:
unique_match_ids = np.unique(match_id)

mask_matrix   = match_id == unique_match_ids[:, None] # broadcast comparison
total_runs_pm = mask_matrix @ total_runs # matrix-vector product
runs_per_match = list(zip(unique_match_ids, total_runs_pm))

print("Sample — Total Runs per Match (first 10):")
print(f"{'Match ID':>10}  {'Total Runs':>12}")
print("-" * 25)
for mid, runs in runs_per_match[:10]:
    print(f"{mid:>10}  {runs:>12}")

Sample — Total Runs per Match (first 10):
  Match ID    Total Runs
-------------------------
    335982           304
    335983           447
    335984           261
    335985           331
    335986           222
    335987           334
    335988           285
    335989           410
    335990           431
    335991           298


# Task 2 : Top 5 batters

In [9]:
unique_batters = np.unique(batter)
batter_mask_matrix = batter == unique_batters[:, None]  # broadcast
batter_total_runs  = batter_mask_matrix @ batsman_runs  # dot product

# Sort descending → top 5
sorted_idx = np.argsort(batter_total_runs)[::-1] # can also do 
top5_batters = [(unique_batters[i], batter_total_runs[i]) for i in sorted_idx[:5]]

print("Top 5 Batters by Total Runs:")
print(f"{'Rank':<6} {'Player':<30} {'Total Runs':>12}")
print("-" * 50)
for rank, (player, runs) in enumerate(top5_batters, 1):
    print(f"{rank:<6} {player:<30} {runs:>12}")

Top 5 Batters by Total Runs:
Rank   Player                           Total Runs
--------------------------------------------------
1      V Kohli                                8014
2      S Dhawan                               6769
3      RG Sharma                              6630
4      DA Warner                              6567
5      SK Raina                               5536


# Task 3 : Strike Rate 

In [10]:
# Reuse batter_mask_matrix from Task 2
# Balls faced = number of rows per batter (sum of boolean mask along axis=1)
balls_faced    = batter_mask_matrix.sum(axis=1) # vectorised count
strike_rate    = np.where(
    balls_faced > 0,
    (batter_total_runs / balls_faced) * 100,
    0.0
)
# Top 10 by strike rate (min 100 balls faced to qualify)
qualified_mask = balls_faced >= 100
qualified_idx  = np.where(qualified_mask)[0]
sr_sorted_idx  = qualified_idx[np.argsort(strike_rate[qualified_idx])[::-1]]

print("Top 10 Batters by Strike Rate (min 100 balls faced):")
print(f"{'Rank':<6} {'Player':<30} {'Balls':>8} {'Runs':>8} {'SR':>10}")
print("-" * 65)
for rank, i in enumerate(sr_sorted_idx[:10], 1):
    print(f"{rank:<6} {unique_batters[i]:<30} {balls_faced[i]:>8} "
          f"{batter_total_runs[i]:>8} {strike_rate[i]:>10.2f}")

Top 10 Batters by Strike Rate (min 100 balls faced):
Rank   Player                            Balls     Runs         SR
-----------------------------------------------------------------
1      J Fraser-McGurk                     150      330     220.00
2      WG Jacks                            133      230     172.93
3      PD Salt                             385      653     169.61
4      T Stubbs                            239      405     169.46
5      TM Head                             458      772     168.56
6      AD Russell                         1515     2488     164.22
7      BCJ Cutting                         146      238     163.01
8      H Klaasen                           613      993     161.99
9      Ramandeep Singh                     106      170     160.38
10     Ashutosh Sharma                     118      189     160.17


# Task 4 : Economy Rate

In [12]:
unique_bowlers = np.unique(bowler)

bowler_mask = bowler == unique_bowlers[:, None]
runs_conceded = bowler_mask @ total_runs          # dot with total_runs column

balls_bowled  = bowler_mask.sum(axis=1)
# Overs = balls / 6
overs_bowled  = balls_bowled / 6.0

economy_rate  = np.where(
    overs_bowled > 0,
    runs_conceded / overs_bowled,
    0.0
)

# Top 10 most economical (min 10 overs bowled)
eco_qualified = balls_bowled >= 60      # 10 overs minimum
eco_q_idx     = np.where(eco_qualified)[0]
eco_sorted    = eco_q_idx[np.argsort(economy_rate[eco_q_idx])]

print("Top 10 Most Economical Bowlers (min 10 overs bowled):")
print(f"{'Rank':<6} {'Bowler':<30} {'Balls':>7} {'Overs':>7} {'Runs':>7} {'Economy':>10}")
print("-" * 72)
for rank, i in enumerate(eco_sorted[:10], 1):
    print(f"{rank:<6} {unique_bowlers[i]:<30} {balls_bowled[i]:>7} "
          f"{overs_bowled[i]:>7.1f} {runs_conceded[i]:>7} {economy_rate[i]:>10.2f}")

Top 10 Most Economical Bowlers (min 10 overs bowled):
Rank   Bowler                           Balls   Overs    Runs    Economy
------------------------------------------------------------------------
1      Sohail Tanvir                      265    44.2     275       6.23
2      A Chandila                         234    39.0     245       6.28
3      FH Edwards                         150    25.0     160       6.40
4      JW Hastings                         61    10.2      66       6.49
5      SMSM Senanayake                    195    32.5     211       6.49
6      MJ Clarke                           66    11.0      72       6.55
7      SM Pollock                         280    46.7     307       6.58
8      SM Harwood                          67    11.2      74       6.63
9      A Kumble                           983   163.8    1089       6.65
10     GD McGrath                         329    54.8     366       6.67


# Task 5 : Average runs per over 

In [13]:
over_numbers = np.arange(0, 20)

over_mask    = over == over_numbers[:, None]        # broadcast
runs_per_ov  = over_mask @ batsman_runs             # sum of runs
balls_per_ov = over_mask.sum(axis=1)               # count of deliveries

avg_runs_per_over = np.where(balls_per_ov > 0, runs_per_ov / balls_per_ov * 6, 0.0)

print("Average Runs per Over (across all matches):")
print(f"{'Over':>6} {'Avg Runs/Over':>15}  {'Bar'}")
print("-" * 50)
for ov, avg in zip(over_numbers, avg_runs_per_over):
    bar = '█' * int(avg)
    print(f"{ov:>6} {avg:>15.2f}  {bar}")

Average Runs per Over (across all matches):
  Over   Avg Runs/Over  Bar
--------------------------------------------------
     0            5.35  █████
     1            6.49  ██████
     2            7.47  ███████
     3            7.74  ███████
     4            7.86  ███████
     5            7.84  ███████
     6            6.26  ██████
     7            6.82  ██████
     8            7.13  ███████
     9            7.03  ███████
    10            7.28  ███████
    11            7.42  ███████
    12            7.44  ███████
    13            7.73  ███████
    14            8.00  ████████
    15            8.20  ████████
    16            8.53  ████████
    17            9.02  █████████
    18            9.34  █████████
    19           10.02  ██████████


# Task 6 : Boundary analysis

In [14]:
fours_mask  = batsman_runs == 4
sixes_mask  = batsman_runs == 6

boundary_mask = fours_mask | sixes_mask

total_fours = int(np.sum(fours_mask))
total_sixes = int(np.sum(sixes_mask))

print(f"Total Fours : {total_fours:,}")
print(f"Total Sixes : {total_sixes:,}")
print(f"Total Boundaries : {total_fours + total_sixes:,}\n")

unique_teams  = np.unique(batting_team)
team_mask_mat = batting_team == unique_teams[:, None]   # (n_teams, n_deliveries)

team_fours    = team_mask_mat @ fours_mask.astype(int)
team_sixes    = team_mask_mat @ sixes_mask.astype(int)
team_bounds   = team_fours + team_sixes

sorted_team_idx = np.argsort(team_bounds)[::-1]

print("Boundaries by Batting Team (Top 10):")
print(f"{'Team':<40} {'4s':>6} {'6s':>6} {'Total':>8}")
print("-" * 63)
for i in sorted_team_idx[:10]:
    print(f"{unique_teams[i]:<40} {team_fours[i]:>6} {team_sixes[i]:>6} {team_bounds[i]:>8}")

best_team_idx = np.argmax(team_bounds)
print(f"\n🏆 Most Boundaries: {unique_teams[best_team_idx]} ({team_bounds[best_team_idx]} boundaries)")

Total Fours : 29,850
Total Sixes : 13,051
Total Boundaries : 42,901

Boundaries by Batting Team (Top 10):
Team                                         4s     6s    Total
---------------------------------------------------------------
Mumbai Indians                             3637   1685     5322
Royal Challengers Bengaluru                3378   1653     5031
Kolkata Knight Riders                      3461   1495     4956
Punjab Kings                               3426   1515     4941
Delhi Capitals                             3508   1351     4859
Chennai Super Kings                        3196   1509     4705
Rajasthan Royals                           3091   1237     4328
Sunrisers Hyderabad                        2405   1042     3447
Deccan Chargers                             957    400     1357
Gujarat Titans                              691    271      962

🏆 Most Boundaries: Mumbai Indians (5322 boundaries)


# Task 7 : Death Over analysis

In [15]:
death_over_mask = (over >= 15) & (over <= 19)
total_death_runs = int(np.sum(batsman_runs[death_over_mask]))
print(f"Total Runs in Death Overs (16–20): {total_death_runs:,}")

death_batting_team = batting_team[death_over_mask]
death_batsman_runs = batsman_runs[death_over_mask]

death_team_mask  = death_batting_team == unique_teams[:, None]
death_team_runs  = death_team_mask @ death_batsman_runs

death_sorted_idx = np.argsort(death_team_runs)[::-1]

print("\nTop 10 Teams — Death Over Runs:")
print(f"{'Team':<40} {'Death Runs':>12}")
print("-" * 54)
for i in death_sorted_idx[:10]:
    print(f"{unique_teams[i]:<40} {death_team_runs[i]:>12}")

best_death_idx = np.argmax(death_team_runs)
print(f"\n🏆 Highest Death-Over Scorer: {unique_teams[best_death_idx]} ({death_team_runs[best_death_idx]} runs)")

Total Runs in Death Overs (16–20): 88,907

Top 10 Teams — Death Over Runs:
Team                                       Death Runs
------------------------------------------------------
Mumbai Indians                                  11245
Royal Challengers Bengaluru                     10741
Chennai Super Kings                             10531
Punjab Kings                                     9903
Delhi Capitals                                   9661
Kolkata Knight Riders                            9567
Rajasthan Royals                                 8688
Sunrisers Hyderabad                              7260
Deccan Chargers                                  2980
Gujarat Titans                                   2124

🏆 Highest Death-Over Scorer: Mumbai Indians (11245 runs)


# Task 8 : Highest scoring match

In [21]:
# Reuse total_runs_pm from Task 1
max_runs_idx    = np.argmax(total_runs_pm)
highest_match   = unique_match_ids[max_runs_idx]
highest_runs    = total_runs_pm[max_runs_idx]

print(f"Highest Scoring Match:")
print(f"  Match ID   : {highest_match}")
print(f"  Total Runs : {highest_runs}")


Highest Scoring Match:
  Match ID   : 1426268
  Total Runs : 549


# Task 9 : Match Winner Approximation

In [23]:
composite_keys = np.char.add(
    np.char.add(match_id.astype(str), '__'),
    batting_team
)

unique_keys, inv_idx = np.unique(composite_keys, return_inverse=True)
team_match_runs = np.bincount(inv_idx, weights=batsman_runs)
# Parse unique_keys back to match_id and team
split_keys   = np.array([k.split('__') for k in unique_keys])  # (N, 2)
key_match_id = split_keys[:, 0].astype(int)
key_team     = split_keys[:, 1]

winners = []
for mid in unique_match_ids:
    idx_for_match = np.where(key_match_id == mid)[0]
    if len(idx_for_match) < 2:
        continue
    runs_for_teams = team_match_runs[idx_for_match]
    winner_idx     = np.argmax(runs_for_teams)
    winners.append((mid, key_team[idx_for_match[winner_idx]], int(runs_for_teams[winner_idx])))

winners_arr = np.array(winners, dtype=object)

print("Approximate Match Winners (by total runs), first 15:")
print(f"{'Match ID':>10} {'Winning Team':<40} {'Runs':>8}")
print("-" * 62)
for mid, team, runs in winners_arr[:15]:
    print(f"{str(mid):>10} {team:<40} {str(runs):>8}")

Approximate Match Winners (by total runs), first 15:
  Match ID Winning Team                                 Runs
--------------------------------------------------------------
    335982 Kolkata Knight Riders                         205
    335983 Chennai Super Kings                           234
    335984 Delhi Capitals                                122
    335985 Royal Challengers Bengaluru                   161
    335986 Deccan Chargers                               100
    335987 Punjab Kings                                  162
    335988 Deccan Chargers                               137
    335989 Chennai Super Kings                           190
    335990 Rajasthan Royals                              210
    335991 Punjab Kings                                  175
    335992 Rajasthan Royals                              135
    335993 Kolkata Knight Riders                         139
    335994 Deccan Chargers                               146
    335995 Punjab Kings       

# Task 10 : Toss impact analysis

In [44]:
match_winners = []

for i in unique_match_ids[:10]:
    match_mask = match_id == i
    teams = np.unique(batting_team[match_mask])
    
    if len(teams) != 2:
        continue
    
    team1, team2 = teams
    team1_runs = np.sum(total_runs[match_mask & (batting_team == team1)])
    team2_runs = np.sum(total_runs[match_mask & (batting_team == team2)])
    
    if team1_runs > team2_runs:
        winner = team1
    elif team2_runs > team1_runs:
        winner = team2
    else:
        winner = 'Tie'
        
    match_winners.append((i, team1, team1_runs, team2, team2_runs, winner))
print(match_winners)

[(np.int64(335982), np.str_('Kolkata Knight Riders'), np.int64(222), np.str_('Royal Challengers Bengaluru'), np.int64(82), np.str_('Kolkata Knight Riders')), (np.int64(335983), np.str_('Chennai Super Kings'), np.int64(240), np.str_('Punjab Kings'), np.int64(207), np.str_('Chennai Super Kings')), (np.int64(335984), np.str_('Delhi Capitals'), np.int64(132), np.str_('Rajasthan Royals'), np.int64(129), np.str_('Delhi Capitals')), (np.int64(335985), np.str_('Mumbai Indians'), np.int64(165), np.str_('Royal Challengers Bengaluru'), np.int64(166), np.str_('Royal Challengers Bengaluru')), (np.int64(335986), np.str_('Deccan Chargers'), np.int64(110), np.str_('Kolkata Knight Riders'), np.int64(112), np.str_('Kolkata Knight Riders')), (np.int64(335987), np.str_('Punjab Kings'), np.int64(166), np.str_('Rajasthan Royals'), np.int64(168), np.str_('Rajasthan Royals')), (np.int64(335988), np.str_('Deccan Chargers'), np.int64(142), np.str_('Delhi Capitals'), np.int64(143), np.str_('Delhi Capitals')), (n

# Task 11 : Match ScoreCard

In [46]:

print("=" * 60)
print("          MATCH SCORECARDS (First 10 Matches)")
print("=" * 60)

for mid in unique_match_ids[:10]:
    idx_for_match = np.where(key_match_id == mid)[0]
    print(f"\n{'─'*50}")
    print(f" Match {mid}")
    print(f"{'─'*50}")

    if len(idx_for_match) == 0:
        print("  No data available.")
        continue

    runs_arr  = team_match_runs[idx_for_match]
    teams_arr = key_team[idx_for_match]

    # Sort by runs descending (team 1 = higher scorer = likely winner)
    order = np.argsort(runs_arr)[::-1]

    for rank, i in enumerate(order):
        label = "🏆 Winner (approx.)" if rank == 0 else "  Runner-up        "
        print(f"  {label}: {teams_arr[i]:<35} {int(runs_arr[i]):>5} runs")

print(f"\n{'─'*50}")
print(f"Total matches in dataset: {len(unique_match_ids)}")

          MATCH SCORECARDS (First 10 Matches)

──────────────────────────────────────────────────
 Match 335982
──────────────────────────────────────────────────
  🏆 Winner (approx.): Kolkata Knight Riders                 205 runs
    Runner-up        : Royal Challengers Bengaluru            63 runs

──────────────────────────────────────────────────
 Match 335983
──────────────────────────────────────────────────
  🏆 Winner (approx.): Chennai Super Kings                   234 runs
    Runner-up        : Punjab Kings                          196 runs

──────────────────────────────────────────────────
 Match 335984
──────────────────────────────────────────────────
  🏆 Winner (approx.): Rajasthan Royals                      122 runs
    Runner-up        : Delhi Capitals                        122 runs

──────────────────────────────────────────────────
 Match 335985
──────────────────────────────────────────────────
  🏆 Winner (approx.): Royal Challengers Bengaluru           161 runs
